# SQLite Manual Analysis

Use this notebook to inspect the SQLite database created by the Rekordbox ingestion pipeline.

In [1]:
from pathlib import Path
import os
import sqlite3

try:
    from dotenv import load_dotenv
except ImportError:
    load_dotenv = None

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if load_dotenv:
    load_dotenv(PROJECT_ROOT / ".env")

db_path = Path(os.getenv("SQLITE_DB_PATH", "data/rekordbox_tracks.sqlite3"))
if not db_path.is_absolute():
    db_path = PROJECT_ROOT / db_path

print(f"Project root: {PROJECT_ROOT}")
print(f"SQLite database: {db_path}")
print(f"Database exists: {db_path.exists()}")

Project root: /Users/zacurbiztondo/dj-library-helper
SQLite database: /Users/zacurbiztondo/dj-library-helper/data/rekordbox_tracks.sqlite3
Database exists: True


## List Tables

In [2]:
if not db_path.exists():
    raise FileNotFoundError(f"SQLite database not found: {db_path}")

conn = sqlite3.connect(db_path)
conn.row_factory = sqlite3.Row

tables = conn.execute(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name
    """
).fetchall()

table_names = [row["name"] for row in tables]
print("Tables:")
for table_name in table_names:
    print(f"- {table_name}")

table_names

Tables:
- rekordbox_tracks


['rekordbox_tracks']

## Print Schemas And Row Counts

In [3]:
for table_name in table_names:
    print(f"\n=== {table_name} ===")
    row_count = conn.execute(f'SELECT COUNT(*) AS count FROM "{table_name}"').fetchone()["count"]
    print(f"Rows: {row_count}")
    print("Columns:")
    for column in conn.execute(f'PRAGMA table_info("{table_name}")').fetchall():
        print(
            f"- {column['name']} {column['type']} "
            f"nullable={not bool(column['notnull'])} pk={bool(column['pk'])}"
        )


=== rekordbox_tracks ===
Rows: 289
Columns:
- rekordbox_track_id INTEGER nullable=False pk=True
- title TEXT nullable=True pk=False
- artist TEXT nullable=True pk=False
- album TEXT nullable=True pk=False
- genre TEXT nullable=True pk=False
- bpm FLOAT nullable=True pk=False
- key VARCHAR(64) nullable=True pk=False
- rating INTEGER nullable=True pk=False
- comments TEXT nullable=True pk=False
- duration FLOAT nullable=True pk=False
- date_added DATETIME nullable=True pk=False
- file_path TEXT nullable=True pk=False
- playlist_name TEXT nullable=False pk=False


## Load Rekordbox Tracks

In [4]:
try:
    import pandas as pd
except ImportError as exc:
    raise ImportError("Install pandas to use the analysis cells: pip install pandas") from exc

tracks = pd.read_sql_query("SELECT * FROM rekordbox_tracks", conn)
print(f"Loaded {len(tracks)} tracks")
tracks.head(20)

Loaded 289 tracks


,rekordbox_track_id,title,artist,album,genre,bpm,key,rating,comments,duration,date_added,file_path,playlist_name
0,257614,baby again.. (original mix),"four tet, skrillex, fred again..",Baby again..,Bass House,127.00,Gm,0,,319.0,2024-11-09 00:00:00.000000,Users/zacurbiztondo/Desktop/DJ-Music/Baby agai...,Z
1,549049,go dj [explicit],lil wayne,Tha Carter [Explicit],Rap & Hip-Hop,79.00,Em,0,Amazon.com Song ID: 202366200,281.0,2024-11-09 00:00:00.000000,Users/zacurbiztondo/Desktop/DJ-Music/Rap/Go DJ...,Z
2,786655,murdah (isoxo remix),knock2,niteharts,Trap / Future Bass,145.00,Dm,0,,172.0,2026-03-31 00:00:00.000000,Users/zacurbiztondo/Downloads/beatport_tracks_...,Z
3,1112161,on my mind (original mix),"diplo, sidepiece",On My Mind,Bass House,123.00,Gm,0,,189.0,2024-11-09 00:00:00.000000,Users/zacurbiztondo/Desktop/DJ-Music/On My Min...,Z
4,1500026,tears,"skrillex, joker, & sleepnet",,,140.00,Fm,0,https://www.youtube.com/watch?,185.0,2026-05-14 00:00:00.000000,"Users/zacurbiztondo/Desktop/DJ-Music/Skrillex,...",Z
5,1516396,"madonna,the weeknd, playboi carti - popular [w...",NaN,,,99.00,Bbm,0,,216.0,2024-11-09 00:00:00.000000,"Users/zacurbiztondo/Desktop/DJ-Music/Madonna,T...",Z
6,2967406,just jammin' (original mix),gramatik,Coffee Shop Selection,Electronica,90.00,Ebm,0,,389.0,2025-11-28 00:00:00.000000,Users/zacurbiztondo/Downloads/beatport_tracks_...,Z
7,7208611,i don't like,chief keef,Still Rich,Drill Rap,131.99,F#m,0,,294.0,2026-05-14 00:00:00.000000,Users/zacurbiztondo/Desktop/DJ-Music/I Don't L...,Z
8,8784213,popular (sped up),NaN,,,114.84,Abm,0,,187.0,2024-11-09 00:00:00.000000,Users/zacurbiztondo/Desktop/DJ-Music/Popular (...,Z
9,9901726,skank n flex (original mix),"wax motif, scrufizzer, taiki nulight",House of Wax,Bass House,130.00,Fm,0,,266.0,2025-01-07 00:00:00.000000,Users/zacurbiztondo/Desktop/DJ-Music/Wax Motif...,Z


## Quick Manual Analysis

In [5]:
summary = {
    "tracks": len(tracks),
    "artists": tracks["artist"].nunique(dropna=True) if "artist" in tracks else None,
    "genres": tracks["genre"].nunique(dropna=True) if "genre" in tracks else None,
    "playlists": tracks["playlist_name"].nunique(dropna=True) if "playlist_name" in tracks else None,
    "avg_bpm": tracks["bpm"].mean() if "bpm" in tracks else None,
}
summary

{'tracks': 289,
 'artists': 201,
 'genres': 35,
 'playlists': 1,
 'avg_bpm': np.float64(124.8187543252595)}

In [6]:
if "playlist_name" in tracks:
    display(tracks["playlist_name"].value_counts().rename_axis("playlist_name").reset_index(name="tracks"))

if "genre" in tracks:
    display(tracks["genre"].value_counts(dropna=False).head(20).rename_axis("genre").reset_index(name="tracks"))

if "artist" in tracks:
    display(tracks["artist"].value_counts(dropna=False).head(20).rename_axis("artist").reset_index(name="tracks"))

,playlist_name,tracks
0,Z,289


,genre,tracks
0,,50
1,Tech House,37
2,Bass House,29
3,Dance / Electro Pop,28
4,House,17
5,Electronica,14
6,Dance / Pop,13
7,Rap & Hip-Hop,9
8,Rap,8
9,Trap / Future Bass,7


,artist,tracks
0,NaN,30
1,knock2,12
2,isoxo,7
3,boris brejcha,5
4,gramatik,4
5,mau p,4
6,funk tribu,4
7,nicolas julian,3
8,rezz,3
9,biscits,3
